# Plant Disease Classifier: Colab GPU

Use a Google Colab GPU (Tesla T4) to:
1. Train or load the PlantVillage classifier.
2. **Option 1 (Shareable Web App)**: Run the Streamlit web app directly on Colab and get a public shareable HTTPS link.
3. **Option 2 (Remote GPU API)**: Expose a FastAPI GPU endpoint so your local Streamlit in VS Code (or Streamlit Cloud) offloads inference to Colab.

Make sure to select **Runtime > Change runtime type > T4 GPU** before running.

In [ ]:
import os
import re
import shutil
import subprocess
import sys
import time
from pathlib import Path

import torch

if not torch.cuda.is_available():
    raise RuntimeError('Select a GPU runtime in Colab (Runtime > Change runtime type > T4 GPU), then rerun this cell.')
print(f'GPU: {torch.cuda.get_device_name(0)}')
print(f'PyTorch: {torch.__version__} | CUDA: {torch.version.cuda}')

## Get the project

Clones the repository and installs the required dependencies (including Streamlit and FastAPI).

In [ ]:
PROJECT_DIR = Path('/content/plant-disease-classifier')
REPOSITORY_URL = 'https://github.com/afnanmengal/plant-disease-classifier.git'
if not PROJECT_DIR.exists():
    subprocess.run(['git', 'clone', REPOSITORY_URL, str(PROJECT_DIR)], check=True)
else:
    subprocess.run(['git', '-C', str(PROJECT_DIR), 'pull'], check=False)

sys.path.insert(0, str(PROJECT_DIR))
sys.path.insert(0, str(PROJECT_DIR / 'src'))
colab_requirements = PROJECT_DIR / 'requirements-colab.txt'
install_command = [sys.executable, '-m', 'pip', 'install', '-q', '-r', str(colab_requirements)]
subprocess.run(install_command, check=True)
print(f'Project ready: {PROJECT_DIR}')

## Model checkpoint (Load or Train)

If you already have a trained checkpoint, upload it to `/content/plant-disease-classifier/models/plant_disease_resnet18.pt` or restore it from Google Drive. If not found, this cell downloads the PlantVillage dataset and trains the ResNet18 model using Colab's GPU.

In [ ]:
CHECKPOINT = PROJECT_DIR / 'models' / 'plant_disease_resnet18.pt'
CHECKPOINT.parent.mkdir(parents=True, exist_ok=True)

# Check if checkpoint is present (uploaded or cached)
if CHECKPOINT.exists() and CHECKPOINT.stat().st_size > 1024:
    print(f'Existing checkpoint found at {CHECKPOINT} ({CHECKPOINT.stat().st_size / (1024*1024):.1f} MB). Skipping training.')
else:
    print('Checkpoint not found. Downloading PlantVillage dataset and training on Colab GPU...')
    import kagglehub
    from scripts.download_dataset import find_class_directory
    from plant_disease.training import train_model

    dataset_cache = kagglehub.dataset_download('emmarex/plantdisease')
    DATA_DIR = find_class_directory(dataset_cache)
    EPOCHS = 5

    history = train_model(
        DATA_DIR,
        CHECKPOINT,
        epochs=EPOCHS,
        batch_size=32,
        pretrained=True,
        num_workers=2,
        device='cuda',
    )
    print(f'Training complete! Checkpoint saved: {CHECKPOINT}')
    print(history)

## Back up the checkpoint (Optional)

Mount Google Drive to persist the model weights across Colab session restarts.

In [ ]:
from google.colab import drive

drive.mount('/content/drive')
DRIVE_DIR = Path('/content/drive/MyDrive/plant-disease-classifier')
DRIVE_DIR.mkdir(parents=True, exist_ok=True)
DRIVE_CHECKPOINT = DRIVE_DIR / 'plant_disease_resnet18.pt'
shutil.copy2(CHECKPOINT, DRIVE_CHECKPOINT)
print(f'Checkpoint backed up to {DRIVE_CHECKPOINT}')

## --- OPTION 1: Run Streamlit directly on Colab (Shareable Public Web App) ---

This runs the complete Streamlit UI directly on Google Colab using its T4 GPU. Cloudflare Tunnel generates an instant public HTTPS link that you can share with anyone.

In [ ]:
# Install Cloudflare tunnel if needed
cloudflared_deb = '/tmp/cloudflared.deb'
if not Path(cloudflared_deb).exists():
    download_command = (
        'wget -q '
        'https://github.com/cloudflare/cloudflared/releases/latest/download/'
        f'cloudflared-linux-amd64.deb -O {cloudflared_deb} '
        f'&& dpkg -i {cloudflared_deb}'
    )
    subprocess.run(['bash', '-lc', download_command], check=True)

# Stop any existing processes
subprocess.run(['pkill', '-f', 'streamlit'], stderr=subprocess.DEVNULL)
subprocess.run(['pkill', '-f', 'cloudflared'], stderr=subprocess.DEVNULL)
time.sleep(1)

# Start Streamlit app
st_log = open('/content/streamlit.log', 'w')
st_proc = subprocess.Popen(
    [
        sys.executable,
        '-m',
        'streamlit',
        'run',
        'app.py',
        '--server.port',
        '8501',
        '--server.headless',
        'true',
    ],
    cwd=PROJECT_DIR,
    stdout=st_log,
    stderr=subprocess.STDOUT,
)

# Start Cloudflare tunnel targeting Streamlit port 8501
tunnel_st_log = open('/content/cloudflared_streamlit.log', 'w')
tunnel_st_proc = subprocess.Popen(
    ['cloudflared', 'tunnel', '--url', 'http://127.0.0.1:8501', '--no-autoupdate'],
    stdout=tunnel_st_log,
    stderr=subprocess.STDOUT,
)

STREAMLIT_URL = None
for _ in range(60):
    if Path('/content/cloudflared_streamlit.log').exists():
        log_text = Path('/content/cloudflared_streamlit.log').read_text(errors='replace')
        match = re.search(r'https://[a-z0-9-]+\.trycloudflare\.com', log_text)
        if match:
            STREAMLIT_URL = match.group(0)
            break
    if tunnel_st_proc.poll() is not None:
        raise RuntimeError(Path('/content/cloudflared_streamlit.log').read_text()[-2000:])
    time.sleep(1)

if STREAMLIT_URL:
    print('=' * 65)
    print('🌿 Streamlit App is LIVE on Google Colab GPU!')
    print(f'🚀 Shareable Public Link: {STREAMLIT_URL}')
    print('=' * 65)
    print('Share this link with anyone to test leaf disease classification!')
else:
    raise TimeoutError('Could not obtain Cloudflare tunnel URL. Check /content/cloudflared_streamlit.log')

## --- OPTION 2: Run FastAPI GPU backend (for Local VS Code or Streamlit Cloud) ---

Run an authenticated FastAPI service on Colab GPU. Then configure the printed Tunnel URL in your local VS Code Streamlit app (in the sidebar or `.streamlit/secrets.toml`) to offload heavy inference to Colab.

In Colab's Secrets panel (key icon on the left), add `PLANT_DISEASE_API_KEY` with any secret password you choose.

In [ ]:
from google.colab import userdata

api_key = userdata.get('PLANT_DISEASE_API_KEY')
if not api_key:
    raise RuntimeError('Add PLANT_DISEASE_API_KEY in the Colab Secrets panel first.')

service_env = os.environ.copy()
service_env['PLANT_DISEASE_API_KEY'] = api_key
service_env['PLANT_DISEASE_CHECKPOINT'] = str(CHECKPOINT)
api_log = open('/content/plant-disease-api.log', 'w')
api_command = [
    sys.executable,
    '-m',
    'uvicorn',
    'plant_disease.colab_api:app',
    '--host',
    '127.0.0.1',
    '--port',
    '8000',
]
api_process = subprocess.Popen(
    api_command,
    cwd=PROJECT_DIR,
    env=service_env,
    stdout=api_log,
    stderr=subprocess.STDOUT,
)

import requests
for _ in range(60):
    try:
        health = requests.get('http://127.0.0.1:8000/health', timeout=2)
        health.raise_for_status()
        break
    except requests.RequestException:
        if api_process.poll() is not None:
            raise RuntimeError(Path('/content/plant-disease-api.log').read_text())
        time.sleep(1)
else:
    raise TimeoutError('Colab GPU API did not become ready.')
print(f'GPU API ready: {health.json()}')

In [ ]:
cloudflared_deb = '/tmp/cloudflared.deb'
if not Path(cloudflared_deb).exists():
    download_command = (
        'wget -q '
        'https://github.com/cloudflare/cloudflared/releases/latest/download/'
        f'cloudflared-linux-amd64.deb -O {cloudflared_deb} '
        f'&& dpkg -i {cloudflared_deb}'
    )
    subprocess.run(['bash', '-lc', download_command], check=True)
tunnel_log = open('/content/cloudflared.log', 'w')
tunnel_process = subprocess.Popen(
    ['cloudflared', 'tunnel', '--url', 'http://127.0.0.1:8000', '--no-autoupdate'],
    stdout=tunnel_log,
    stderr=subprocess.STDOUT,
)

TUNNEL_URL = None
for _ in range(60):
    log_text = Path('/content/cloudflared.log').read_text(errors='replace')
    match = re.search(r'https://[a-z0-9-]+\.trycloudflare\.com', log_text)
    if match:
        TUNNEL_URL = match.group(0)
        break
    if tunnel_process.poll() is not None:
        raise RuntimeError(log_text[-3000:])
    time.sleep(1)
if not TUNNEL_URL:
    raise TimeoutError('Cloudflare tunnel URL did not appear in the log.')
print(f'colab_api_url = {TUNNEL_URL!r}')
print('Paste this URL and your token in your local VS Code Streamlit sidebar under "Colab GPU Endpoint Settings"!')